In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://entire-causal-dollhouse.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://entire-causal-dollhouse.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology），簡稱明新科大，是一所位於臺灣新竹縣新豐鄉的私立科技大學。學校秉持「堅毅、求新、創造」的校訓精神，以培育「跨域整合、務實創新與全人學習」的專業人才為目標。

**歷史沿革**
明新科技大學的創校歷史可追溯至1966年成立的「明新工業專科學校」。1997年，經教育部核准改制為「明新技術學院」並附設專科部。歷經多年的發展與努力，學校於2002年9月正式升格並更名為「明新科技大學」。2019年3月，學校再次更名為「明新學校財團法人明新科技大學」。

**地理位置與校園**
明新科技大學座落於新竹縣新豐鄉，佔地逾二十三公頃（另有資料顯示逾三十公頃），緊鄰省道縱貫線與中山高速公路，交通便捷。其地理位置優越，鄰近新竹科學園區與新竹工業區，享有豐富的產業資源，為學校發展產學合作提供了絕佳條件。

**學術單位與特色**
目前，明新科技大學設有半導體學院、工程學院、管理學院、民生學院、人文與設計學院、共同教育學院等六個學院。學校提供20個學系、2個學位學程（包含1個博士學位學程）以及11個碩士班。在學術發展上，明新科大特別聚焦於半導體、AI、元宇宙、風電綠能等前瞻產業。

明新科大致力於成為一所「一流產業大學」，並發展出「MUST」四大育才特色：多元學習（Multidisciplinary Learning）、全球視野（Universal Perspective）、永續經營（Sustainable Operations）與技術創新（Technological Innovation），旨在引導學生進行跨域學習，成為產業最搶手的技職人才。學校亦積極推動產學合作，其畢業生在業界表現優異，曾榮獲《Cheers》雜誌「企業最愛公私立技職科大調查」中兩項企業最愛，起薪排名私校第一，並優於部分國立科大。學校的就業率也連續三年位居桃竹苗地區第一，全國前五大。此外，明新科大積極推動國際化，擁有為數不少的國際學生，致力於培養學生的國際視野。


In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學現任校長為**呂明峯**教授。

呂明峯教授於2025年2月1日正式上任，接替卸任的林啓瑞校長，成為明新科技大學第11任校長。 他在半導體教育及產業實務方面擁有豐富經驗，並提出「四大核心模組」作為學校未來發展的理念，旨在將明新科大打造成為桃竹苗大矽谷的人才引擎、新南向專班的基地，並活化資源以實現永續校園。


In [ ]:
from flask import Flask, request, abort
import logging
import os
import time
from google.genai import types

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    MessagingApiBlob,
    ReplyMessageRequest,
    TextMessage
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
    FileMessageContent
)

app = Flask(__name__)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
app.logger.setLevel(logging.INFO)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

# 儲存檔案的目錄
UPLOAD_DIR = "/content/uploaded_files"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# 儲存每個使用者的對話 session 和上傳的檔案
user_sessions = {}  # {user_id: {"chat": chat_object, "uploaded_file": gemini_file}}

def get_user_session(user_id):
    """取得或建立使用者的對話 session"""
    if user_id not in user_sessions:
        # 建立新的對話 session
        new_chat = client.chats.create(
            model="gemini-2.5-flash",
            config=GenerateContentConfig(
                system_instruction="你是一個中文的AI助手，請用繁體中文回答。如果使用者有提供參考文件，請根據文件內容回答問題。",
                tools=[google_search_tool],
                response_modalities=["TEXT"],
            )
        )
        user_sessions[user_id] = {
            "chat": new_chat,
            "uploaded_file": None
        }
    return user_sessions[user_id]

def download_line_file(message_id, file_name):
    """從 LINE 下載使用者上傳的檔案"""
    with ApiClient(configuration) as api_client:
        line_bot_blob_api = MessagingApiBlob(api_client)
        file_content = line_bot_blob_api.get_message_content(message_id)

        file_path = os.path.join(UPLOAD_DIR, file_name)

        with open(file_path, 'wb') as f:
            f.write(file_content)

        return file_path

def upload_file_to_gemini(file_path):
    """上傳檔案到 Gemini Files API"""
    uploaded_file = client.files.upload(
        file=file_path,
        config={'display_name': os.path.basename(file_path)}
    )

    # 等待檔案處理完成
    while uploaded_file.state.name == "PROCESSING":
        print("檔案處理中...")
        time.sleep(1)
        uploaded_file = client.files.get(name=uploaded_file.name)

    if uploaded_file.state.name == "FAILED":
        raise Exception("檔案上傳處理失敗")

    return uploaded_file

def query_with_rag(user_id, question):
    """使用 RAG 模式回答問題"""
    session = get_user_session(user_id)
    uploaded_file = session["uploaded_file"]

    if uploaded_file:
        # 有上傳檔案，使用 RAG 模式
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_uri(
                            file_uri=uploaded_file.uri,
                            mime_type=uploaded_file.mime_type
                        ),
                        types.Part.from_text(text=f"請根據上述提供的檔案內容，用繁體中文回答這個問題：{question}")
                    ]
                )
            ]
        )
        return response.text
    else:
        # 沒有上傳檔案，使用一般多輪對話
        response = session["chat"].send_message(message=question)
        return response.text

@app.route("/", methods=['POST'])
def callback():
    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature.")
        abort(400)

    return 'OK'

@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    """處理文字訊息"""
    text = event.message.text
    user_id = event.source.user_id

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]
            try:
                # 使用 RAG 或一般對話
                reply_text = query_with_rag(user_id, prompt)

                # 檢查是否有上傳檔案，加上提示
                session = get_user_session(user_id)
                if session["uploaded_file"]:
                    reply_text = f"📄 [RAG 模式]\n\n{reply_text}"

                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=reply_text)]
                    )
                )
            except Exception as e:
                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=f"❌ 發生錯誤：{str(e)}")]
                    )
                )
        elif text == "清除文件":
            # 清除使用者上傳的檔案
            session = get_user_session(user_id)
            session["uploaded_file"] = None
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="✅ 已清除上傳的文件，恢復一般對話模式。")]
                )
            )
        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="請輸入「AI 問題」來開始對話\n或上傳 TXT/PDF 檔案啟用 RAG 模式")]
                )
            )

@handler.add(MessageEvent, message=FileMessageContent)
def handle_file_message(event):
    """處理使用者上傳的檔案"""
    user_id = event.source.user_id
    file_name = event.message.file_name

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 檢查檔案類型
        if not (file_name.endswith('.txt') or file_name.endswith('.pdf')):
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="⚠️ 目前只支援 TXT 或 PDF 檔案")]
                )
            )
            return

        try:
            # 下載檔案
            file_path = download_line_file(event.message.id, file_name)
            print(f"檔案已下載：{file_path}")

            # 上傳到 Gemini
            uploaded_file = upload_file_to_gemini(file_path)
            print(f"檔案已上傳到 Gemini：{uploaded_file.uri}")

            # 儲存到使用者 session
            session = get_user_session(user_id)
            session["uploaded_file"] = uploaded_file

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"✅ 檔案「{file_name}」上傳成功！\n\n現在您可以輸入「AI 問題」來詢問關於這份文件的問題。\n\n輸入「清除文件」可恢復一般對話模式。")]
                )
            )
        except Exception as e:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"❌ 檔案處理失敗：{str(e)}")]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:__main__:Request body: {"destination":"U2b674d9a6f4fdff79bd2f765a8efdbba","events":[{"type":"message","message":{"type":"text","id":"616611866701463606","quoteToken":"gpias2aN7lHWuURUQ9hszXT_0XKTOCP-9eSwHKBbvaGwnwtu4QgkfPrnC692cImG2Q8le9zNZiNiZp8-IaCldmYbEbG35Tq2qGhM_eQ6C3prR99cbn8glUAHysyD8AzGkX5AyRl0UnI5rl2qHjfuxQ","markAsReadToken":"PN6o-ZyGmgyft73jBHx71Oaz9OAdES32nOLUy1DEsAMJRcr3xJKO2dS0NWWifpBujVDqbG6pgZph_g7QTQKrfCCEoFQJ1cnCOsI7hyDuPCR5HNvNxXJgG7dYMB95ta5SiJT0NDuvoB1Tzb8NY9luOd82ZNwV9Oc3IX87jJwtXXFPhjpg9GHwUK80-s-jbMu8vSGLTqdjEI2ArnCuicqhlA","text":"AI 校長愛吃甚麼"},"webhookEventId":"01KT2W67726GEHJKE8DTJVAY4T","deliveryContext":{"isRedelivery":false},"timestamp":1780360616951,"source":{"type":"user","userId":"Ubbcb50e2459370de34f2b5985837784e"},"replyToken":"0224bdfc52384d83b36d

BODY:  {"destination":"U2b674d9a6f4fdff79bd2f765a8efdbba","events":[{"type":"message","message":{"type":"text","id":"616611866701463606","quoteToken":"gpias2aN7lHWuURUQ9hszXT_0XKTOCP-9eSwHKBbvaGwnwtu4QgkfPrnC692cImG2Q8le9zNZiNiZp8-IaCldmYbEbG35Tq2qGhM_eQ6C3prR99cbn8glUAHysyD8AzGkX5AyRl0UnI5rl2qHjfuxQ","markAsReadToken":"PN6o-ZyGmgyft73jBHx71Oaz9OAdES32nOLUy1DEsAMJRcr3xJKO2dS0NWWifpBujVDqbG6pgZph_g7QTQKrfCCEoFQJ1cnCOsI7hyDuPCR5HNvNxXJgG7dYMB95ta5SiJT0NDuvoB1Tzb8NY9luOd82ZNwV9Oc3IX87jJwtXXFPhjpg9GHwUK80-s-jbMu8vSGLTqdjEI2ArnCuicqhlA","text":"AI 校長愛吃甚麼"},"webhookEventId":"01KT2W67726GEHJKE8DTJVAY4T","deliveryContext":{"isRedelivery":false},"timestamp":1780360616951,"source":{"type":"user","userId":"Ubbcb50e2459370de34f2b5985837784e"},"replyToken":"0224bdfc52384d83b36de537ca3cbcc1","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 00:37:02] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"U2b674d9a6f4fdff79bd2f765a8efdbba","events":[{"type":"message","message":{"type":"file","id":"616611890659590368","markAsReadToken":"SJEncTmgPRANSdRw-NRCyTgDIULffdkNHwyc36T2G5fIJKZKOLjLvr1HvlSd_MnHkDnmkMMAVneH00-EIY5jXrm45fOqDvjK7EGEst5Ttf6veQ61ds4FVi4HHm444cA5zc0OJi4oWO2lrrw733LNy7mEU1er8R6ks1OqwkYuSf6wd1aUlvMww6uJ_JCL5_N-b5CVhXKKviLmH_RqgWNh4w","fileName":"RAG.txt","fileSize":18,"contentProvider":{"type":"line"}},"webhookEventId":"01KT2W6N6BGQWNZC1QNYK850ZF","deliveryContext":{"isRedelivery":false},"timestamp":1780360631238,"source":{"type":"user","userId":"Ubbcb50e2459370de34f2b5985837784e"},"replyToken":"fc684b67f2ce410d9cee73bef7d8819d","mode":"active"}]}


BODY:  {"destination":"U2b674d9a6f4fdff79bd2f765a8efdbba","events":[{"type":"message","message":{"type":"file","id":"616611890659590368","markAsReadToken":"SJEncTmgPRANSdRw-NRCyTgDIULffdkNHwyc36T2G5fIJKZKOLjLvr1HvlSd_MnHkDnmkMMAVneH00-EIY5jXrm45fOqDvjK7EGEst5Ttf6veQ61ds4FVi4HHm444cA5zc0OJi4oWO2lrrw733LNy7mEU1er8R6ks1OqwkYuSf6wd1aUlvMww6uJ_JCL5_N-b5CVhXKKviLmH_RqgWNh4w","fileName":"RAG.txt","fileSize":18,"contentProvider":{"type":"line"}},"webhookEventId":"01KT2W6N6BGQWNZC1QNYK850ZF","deliveryContext":{"isRedelivery":false},"timestamp":1780360631238,"source":{"type":"user","userId":"Ubbcb50e2459370de34f2b5985837784e"},"replyToken":"fc684b67f2ce410d9cee73bef7d8819d","mode":"active"}]}
檔案已下載：/content/uploaded_files/RAG.txt
檔案已上傳到 Gemini：https://generativelanguage.googleapis.com/v1beta/files/wfxy4hgv4550


INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 00:37:14] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"U2b674d9a6f4fdff79bd2f765a8efdbba","events":[{"type":"message","message":{"type":"text","id":"616611901162127548","quoteToken":"xVq6KHjPICLMIj9RsrLc1kOtR0gUJtFlkCW_sKdmZyYu-RTmlOZkUAR5NW7R2BESlFr-4tCweOwbtykEkzt-zSkI2yonZb3LrD7AjcQO6Ci9EuWscdLddQYhOfLaBMqUU_-mscMzoChg3CviAEAMVQ","markAsReadToken":"rexBISTxRkVbf8G_CS4owyZ_iweIvIdCjx8STh2OzPlcNwCAIYGWum-zPydl_B3uWC998Vhyuzl0gw_7ytIyVAF7TRmVb2pxA7U2Tsh5Mjf4jkCS94ngUawpUaQj2Hsl9l0xuuBPZCQG8FDVCxY-gvcdNKZKzKIIAsDACUVY37v_3xFsAlfH1uo8Ik2zWPnseDV_RPsVzqhG5mtoeADOlQ","text":"AI 校長愛吃甚麼"},"webhookEventId":"01KT2W6VCRDP02CRZVBEJK413M","deliveryContext":{"isRedelivery":false},"timestamp":1780360637340,"source":{"type":"user","userId":"Ubbcb50e2459370de34f2b5985837784e"},"replyToken":"1ff12c5769f145709a0085cfcd61dbbb","mode":"active"}]}


BODY:  {"destination":"U2b674d9a6f4fdff79bd2f765a8efdbba","events":[{"type":"message","message":{"type":"text","id":"616611901162127548","quoteToken":"xVq6KHjPICLMIj9RsrLc1kOtR0gUJtFlkCW_sKdmZyYu-RTmlOZkUAR5NW7R2BESlFr-4tCweOwbtykEkzt-zSkI2yonZb3LrD7AjcQO6Ci9EuWscdLddQYhOfLaBMqUU_-mscMzoChg3CviAEAMVQ","markAsReadToken":"rexBISTxRkVbf8G_CS4owyZ_iweIvIdCjx8STh2OzPlcNwCAIYGWum-zPydl_B3uWC998Vhyuzl0gw_7ytIyVAF7TRmVb2pxA7U2Tsh5Mjf4jkCS94ngUawpUaQj2Hsl9l0xuuBPZCQG8FDVCxY-gvcdNKZKzKIIAsDACUVY37v_3xFsAlfH1uo8Ik2zWPnseDV_RPsVzqhG5mtoeADOlQ","text":"AI 校長愛吃甚麼"},"webhookEventId":"01KT2W6VCRDP02CRZVBEJK413M","deliveryContext":{"isRedelivery":false},"timestamp":1780360637340,"source":{"type":"user","userId":"Ubbcb50e2459370de34f2b5985837784e"},"replyToken":"1ff12c5769f145709a0085cfcd61dbbb","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 00:37:19] "POST / HTTP/1.1" 200 -
